# NLP-Based Movie Recommendation System
### Content-Based Recommendation using TF-IDF and Cosine Similarity

This notebook implements **Level 1 (Core)** of the project:

1. Load and clean the TMDB 5000 movie dataset
2. Preprocess text (lowercasing, tokenization, stopword removal, lemmatization)
3. Combine genres + overview + keywords into a single "movie document"
4. Convert movie documents into TF-IDF vectors
5. Accept a free-text user query, vectorize it, and rank movies by cosine similarity
6. Return the Top-N recommended movies


In [15]:
import pandas as pd
import numpy as np
import ast
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

pd.set_option('display.max_colwidth', 100)


## 1. Load the dataset

Using the **TMDB 5000 Movie Dataset** (`tmdb_5000_movies.csv`), which contains `title`, `genres`, `overview`, and `keywords` — everything needed for Level 1.

In [16]:
movies = pd.read_csv('data/tmdb_5000_movies.csv')
print("Shape:", movies.shape)
movies.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'data/tmdb_5000_movies.csv'

In [ ]:
# Keep only the columns we need for the core recommender
movies = movies[['id', 'title', 'genres', 'keywords', 'overview', 'vote_average', 'release_date']]
movies.head(3)


,id,title,genres,keywords,overview,vote_average,release_date
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""sp...","In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, ...",7.2,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic is...","Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of t...",6.9,2007-05-19
2,206647,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret...",A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. Whil...,6.3,2015-10-26


## 2. Clean the data

Drop rows with missing overviews (the most important text field), and reset the index so it lines up with our similarity matrix rows.

In [ ]:
print("Missing values before cleaning:")
print(movies.isnull().sum())

movies = movies.dropna(subset=['overview']).reset_index(drop=True)
movies['keywords'] = movies['keywords'].fillna('[]')
movies['genres'] = movies['genres'].fillna('[]')

print("\nShape after cleaning:", movies.shape)


Missing values before cleaning:
id              0
title           0
genres          0
keywords        0
overview        3
vote_average    0
release_date    1
dtype: int64

Shape after cleaning: (4800, 7)


## 3. Parse the JSON-like columns

`genres` and `keywords` are stored as stringified lists of dicts, e.g.:

```
[{"id": 878, "name": "Science Fiction"}, {"id": 28, "name": "Action"}]
```

We extract just the `name` values.

In [ ]:
def parse_names(json_like_str):
    try:
        items = ast.literal_eval(json_like_str)
        return [d['name'] for d in items]
    except (ValueError, SyntaxError):
        return []

movies['genres_list'] = movies['genres'].apply(parse_names)
movies['keywords_list'] = movies['keywords'].apply(parse_names)

movies[['title', 'genres_list', 'keywords_list']].head(3)


,title,genres_list,keywords_list
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, spa..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, ship..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]"


## 4. Text preprocessing

Standard NLP cleaning pipeline for the `overview` (free-text plot summary):

- Lowercase
- Remove punctuation / non-letters
- Tokenize
- Remove stopwords
- Lemmatize

Genres and keywords are already clean short phrases, so we just lowercase and join them (no need to lemmatize category names like "Science Fiction").

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)          # keep letters only
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

movies['overview_clean'] = movies['overview'].apply(clean_text)
movies[['title', 'overview', 'overview_clean']].head(3)


,title,overview,overview_clean
0,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, ...",century paraplegic marine dispatched moon pandora unique mission becomes torn following order pr...
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of t...",captain barbossa long believed dead come back life headed edge earth turner elizabeth swann noth...
2,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. Whil...,cryptic message bond past sends trail uncover sinister organization battle political force keep ...


## 5. Build the "movie document"

Combine genres + keywords + cleaned overview into one text blob per movie.
Genres and keywords are repeated (weighted) slightly by including them as space-joined tags so they carry
meaningful signal even though they're short compared to the overview.

In [ ]:
def build_document(row):
    genres_text = ' '.join([g.lower().replace(' ', '') for g in row['genres_list']])
    keywords_text = ' '.join([k.lower().replace(' ', '') for k in row['keywords_list']])
    # Repeat genres twice so genre words aren't drowned out by the longer overview
    return f"{genres_text} {genres_text} {keywords_text} {row['overview_clean']}"

movies['document'] = movies.apply(build_document, axis=1)
movies[['title', 'document']].head(3)


,title,document
0,Avatar,action adventure fantasy sciencefiction action adventure fantasy sciencefiction cultureclash fut...
1,Pirates of the Caribbean: At World's End,adventure fantasy action adventure fantasy action ocean drugabuse exoticisland eastindiatradingc...
2,Spectre,action adventure crime action adventure crime spy basedonnovel secretagent sequel mi6 britishsec...


## 6. TF-IDF vectorization

Convert every movie document into a TF-IDF vector.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=15000,   # cap vocabulary size
    ngram_range=(1, 2),   # unigrams + bigrams (e.g. "space exploration")
    min_df=2              # ignore terms that appear in only 1 movie
)

tfidf_matrix = tfidf.fit_transform(movies['document'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


TF-IDF matrix shape: (4800, 15000)
Vocabulary size: 15000


## 7. The recommendation function

- Clean the user's query the same way we cleaned the overviews
- Transform it with the **same fitted** TF-IDF vectorizer (important — don't refit on the query)
- Compute cosine similarity between the query vector and every movie vector
- Return the Top-N most similar movies

In [ ]:
def recommend(query, top_n=5):
    query_clean = clean_text(query)
    query_vec = tfidf.transform([query_clean])

    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_indices = sims.argsort()[::-1][:top_n]

    results = movies.iloc[top_indices][['title', 'genres_list', 'vote_average', 'release_date']].copy()
    results['similarity'] = sims[top_indices]
    results['similarity_%'] = (results['similarity'] * 100).round(1)
    return results.reset_index(drop=True)


## 8. Try it out

In [ ]:
recommend("science fiction space adventure with an emotional story", top_n=5)


,title,genres_list,vote_average,release_date,similarity,similarity_%
0,Gattaca,"[Thriller, Science Fiction, Mystery, Romance]",7.5,1997-09-07,0.259851,26.0
1,Martian Child,[Drama],6.8,2007-11-02,0.254139,25.4
2,Flatliners,"[Drama, Horror, Science Fiction, Thriller]",6.3,1990-08-09,0.214273,21.4
3,Her,"[Romance, Science Fiction, Drama]",7.9,2013-12-18,0.195623,19.6
4,"The Beast from 20,000 Fathoms","[Adventure, Horror, Science Fiction]",6.7,1953-06-13,0.188478,18.8


In [ ]:
recommend("funny movie about friendship and college life", top_n=5)


,title,genres_list,vote_average,release_date,similarity,similarity_%
0,Disaster Movie,"[Action, Comedy]",3.0,2008-08-29,0.234665,23.5
1,Ask Me Anything,"[Drama, Mystery, Thriller]",5.5,2014-04-19,0.209168,20.9
2,Not Cool,[Comedy],3.7,2014-09-23,0.166812,16.7
3,Jackass 3D,"[Comedy, Documentary, Action]",6.4,2010-10-15,0.166134,16.6
4,Grindhouse,"[Thriller, Action, Horror]",6.8,2007-04-06,0.147374,14.7


In [ ]:
recommend("action movie with a strong female character", top_n=5)


,title,genres_list,vote_average,release_date,similarity,similarity_%
0,Disaster Movie,"[Action, Comedy]",3.0,2008-08-29,0.206645,20.7
1,Kung Pow: Enter the Fist,"[Action, Comedy]",6.1,2002-01-25,0.197349,19.7
2,The Transporter Refueled,"[Thriller, Action, Crime]",5.2,2015-09-03,0.178875,17.9
3,Jackass 3D,"[Comedy, Documentary, Action]",6.4,2010-10-15,0.146297,14.6
4,Grindhouse,"[Thriller, Action, Horror]",6.8,2007-04-06,0.138048,13.8


## 9. Quick evaluation — Precision@5

We define a small set of test queries with a manually-judged "relevant genre" for each,
then check how many of the top-5 results actually contain that genre.

This is a simple proxy for Precision@K since we don't have ground-truth relevance labels —
in a real evaluation you (or classmates) would manually judge each recommendation as relevant/not relevant.

In [ ]:
test_queries = [
    ("science fiction space adventure", "Science Fiction"),
    ("romantic comedy", "Romance"),
    ("superhero action movie", "Action"),
    ("horror movie with ghosts", "Horror"),
    ("animated family movie", "Animation"),
    ("crime thriller", "Crime"),
    ("historical drama", "Drama"),
]

def precision_at_k(query, relevant_genre, k=5):
    results = recommend(query, top_n=k)
    hits = results['genres_list'].apply(lambda genres: relevant_genre in genres)
    return hits.sum() / k

scores = []
for query, genre in test_queries:
    p = precision_at_k(query, genre, k=5)
    scores.append(p)
    print(f"Query: {query!r:45s} | Target genre: {genre:16s} | Precision@5 = {p:.2f}")

print(f"\nAverage Precision@5 across {len(test_queries)} test queries: {np.mean(scores):.2f}")


Query: 'science fiction space adventure'             | Target genre: Science Fiction  | Precision@5 = 0.80
Query: 'romantic comedy'                             | Target genre: Romance          | Precision@5 = 0.40
Query: 'superhero action movie'                      | Target genre: Action           | Precision@5 = 0.80
Query: 'horror movie with ghosts'                    | Target genre: Horror           | Precision@5 = 0.80
Query: 'animated family movie'                       | Target genre: Animation        | Precision@5 = 0.20
Query: 'crime thriller'                              | Target genre: Crime            | Precision@5 = 1.00
Query: 'historical drama'                            | Target genre: Drama            | Precision@5 = 0.80

Average Precision@5 across 7 test queries: 0.69


## 10. Save the artifacts

Save the cleaned dataframe, the fitted vectorizer, and the TF-IDF matrix so the Streamlit app (Level 3)
can load them instantly without re-processing the whole dataset.

In [ ]:
import pickle

with open('movies.pkl', 'wb') as f:
    pickle.dump(movies, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

print("Saved: movies.pkl, tfidf_vectorizer.pkl, tfidf_matrix.pkl")


Saved: movies.pkl, tfidf_vectorizer.pkl, tfidf_matrix.pkl


## Next steps

- **Level 2**: add cast/director (needs `tmdb_5000_credits.csv`), weighted similarity across plot/genre/keyword/cast/director, genre + rating filters.
- **Level 3**: wrap `recommend()` in a Streamlit app with a search box, filters, and "why recommended" explanations.
